# Essai : réentraîner le pidgin avec les données financières (Colab)

Même recette que v2 (base jamm55, mêmes réglages, mêmes données) ; seul le poids des phrases d'argent change.
1. Le fichier **`KAMA_pidgin_finance_drive.zip`** doit être dans ton Drive, à la racine de Mon Drive ou dans un sous-dossier de `KAMA/` : la cellule 2 le trouve toute seule. Ne le décompresse pas.
2. **Exécution → Modifier le type d'exécution → GPU T4**.
3. Lance les cellules dans l'ordre. Commence par `VARIANTE = 'propre'`, puis relance tout avec `'brut'` si tu as le temps.
4. Si Colab coupe, relance tout depuis le début (l'entraînement dure 30 à 60 minutes).

## 1. Réglages
`propre` = emplois financiers nettoyés (×4). `brut` = toutes les phrases d'argent du corpus, religion comprise (×2), comme le fichier finance_en_wes.csv.

In [ ]:
MON_DRIVE = '/content/drive/MyDrive'
VARIANTE = 'propre'   # 'propre' ou 'brut'
EPOCHS = 8
DOSSIER = '/content/KAMA/pidgin_finance'   # disque de Colab : plus rapide et plus fiable que Drive

## 2. Drive et données
Le zip est décompressé à chaque exécution, sur le disque de Colab. Si quelque chose manque, la cellule s'arrête avec un message clair.

In [ ]:
import os, glob, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')
zips = glob.glob(MON_DRIVE + '/KAMA_pidgin_finance_drive.zip') + glob.glob(MON_DRIVE + '/KAMA/**/KAMA_pidgin_finance_drive.zip', recursive=True)
assert zips, 'KAMA_pidgin_finance_drive.zip introuvable dans Mon Drive ni dans KAMA/ : mets-le dans Mon Drive'
print('zip utilisé :', zips[0])
shutil.rmtree('/content/KAMA', ignore_errors=True)
with zipfile.ZipFile(zips[0]) as z:
    assert z.testzip() is None, 'zip abîmé : renvoie-le depuis ton ordinateur'
    z.extractall('/content')
for f in (f'data_{VARIANTE}/train.jsonl', f'data_{VARIANTE}/dev.jsonl', f'data_{VARIANTE}/test_soreva.jsonl', '02_entrainer.py'):
    assert os.path.exists(f'{DOSSIER}/{f}'), f'manquant dans le zip : {f}'
print('Données prêtes :', sum(1 for _ in open(f'{DOSSIER}/data_{VARIANTE}/train.jsonl')), 'exemples')

## 3. Installation

In [ ]:
!pip install -q transformers==4.57.6 sentencepiece sacremoses sacrebleu accelerate ctranslate2
!nvidia-smi -L

## 4. Entraînement
Environ 30 à 60 minutes sur un GPU T4. Regarde en fin de sortie la ligne `TEST SOREVA (jamais vu)` : c'est la note à comparer à v2 (chrF 33,0).

In [ ]:
%cd {DOSSIER}
!python 02_entrainer.py --data data_{VARIANTE} --sortie modele_finance_{VARIANTE} --epochs {EPOCHS}

## 5. Conversion et copie sur Drive
Produit `pidgin_en_wes_finance_<variante>.zip` (environ 80 Mo) et le copie dans `Mon Drive/KAMA/pidgin_finance_sorties`. Télécharge-le et mets-le dans `KAMA/models/poids/ct2/` **sous un autre nom** que `pidgin_en_wes` : il ne doit pas remplacer le modèle du serveur avant comparaison.

In [ ]:
!ct2-transformers-converter --model modele_finance_{VARIANTE}/final --output_dir modele_finance_{VARIANTE}/pidgin_en_wes_finance_{VARIANTE} --quantization int8 --copy_files source.spm target.spm --force
!cd modele_finance_{VARIANTE} && zip -qr pidgin_en_wes_finance_{VARIANTE}.zip pidgin_en_wes_finance_{VARIANTE}
os.makedirs(MON_DRIVE + '/KAMA/pidgin_finance_sorties', exist_ok=True)
shutil.copy(f'modele_finance_{VARIANTE}/pidgin_en_wes_finance_{VARIANTE}.zip', MON_DRIVE + '/KAMA/pidgin_finance_sorties')
print(os.listdir(MON_DRIVE + '/KAMA/pidgin_finance_sorties'))